# Troubleshooting Your Code: Roast Toast Coast Café Web Page

**Name:** Anjali &nbsp;|&nbsp; **Program:** IBM SkillsBuild – *Use IBM Bob to Troubleshoot Your Code* &nbsp;|&nbsp; **Date:** 23 October 2026

## Project overview
The fictional *Roast Toast Coast Café* has a one-page website (HTML + CSS + JavaScript) with a welcome message, a tagline, an image and an **Our Menu** button that should reveal menu items. When previewed, two things are broken:

1. The image does not display.
2. Clicking **Our Menu** does not reveal the menu items.

The lab asks the student to use **IBM Bob (Code mode, in VS Code)** to find and fix the errors. This notebook is the code deliverable for that project. It documents the whole troubleshooting workflow in a reproducible way:

| Lab task | Where it is done in this notebook |
|---|---|
| Task 1 – Review scenario | Section 1 (original code) |
| Task 2 – Analyze the code | Section 2 (structured analyzer, grouped by severity and type) |
| Task 3 – Apply corrections | Section 3 (automated fixes + side-by-side diff) |
| Task 4 – Reflect | Sections 4 and 5 (verification + reflection) |


## 0. Setup

In [1]:
import re
import json
import difflib
import textwrap
from collections import Counter
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SEVERITY_ORDER = ["critical", "high", "medium", "low"]
print("Setup complete. Output folder:", OUTPUT_DIR.resolve())

Setup complete. Output folder: /home/claude/work/run/output


## 1. Task 1 – The original (broken) code

The lab provides `Menu.html` (source: <https://github.com/academic-initiative/skillsbuild/blob/main/ibm-bob/troubleshoot-your-code/file/Menu.html>).
The cell below embeds that file so the notebook runs anywhere. **If you place your own copy named `Menu.html` next to this notebook, it is used instead.**

In [2]:
ORIGINAL_EMBEDDED = """<!DOCTYPE html>
<html lang="en">
<head>
<title>Roast Toast Coast Cafe</title>
<style>
@font-face {
    font-family: 'Bogart';
    src: url('https://fonts.cdnfonts.com/s/24702/Bogart-Light.woff') format('woff');
    font-weight: normal;
    font-style: normal;
}

body {
    background-color: black;
    font-family: 'Bogart', sans-serif;
    margin: 0;
    padding: 0;
    display: grid;
    grid-template-columns: 1fr 1fr; /* Two columns of equal width */
    height: 100vh; /* Full height */
}

.content {
    display: flex;
    flex-direction: column;
    justify-content: center;
    align-items: center;
    padding: 20px;
}

.image-container {
    background-image: url('https://images.unsplash.com/photo-1514066558159-fc8c737ef259?q=80&w=2574&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8MHx8fA%3D%3D';
    background-size: cover;
    background-position: center;
    height: 100%;
}

.header-container {
    background-color: rgba(255, 255, 255, 0.1); /* Semi-transparent white */
    border: 1px solid rgba(255, 255, 255, 0.3); /* Light border for a sleek look */
    border-radius: 15px;
    padding: 20px;
    margin-bottom: 20px;
    width: 90%;
    box-shadow: 0px 4px 15px rgba(0, 0, 0, 0.7); /* Soft shadow */
    text-align: center;
    background: wheat;
}

header h1 {
    color: #FFCC00; /* Bright gold for contrast */
    font-size: 3em;
    margin-bottom: 10px;
    text-shadow: 2px 2px 5px rgba(0, 0, 0, 0.8); /* Subtle shadow for depth */
    line-height: 1.2; /* Adjust spacing between lines */
}

header p {
    font-size: 1.3em;
    color: #FFFFFF; /* White for good readability */
    margin: 0;
    text-shadow: 1px 1px 3px rgba(0, 0, 0, 0.6); /* Subtle shadow for depth */
}

/* .menu-section {
    background-color: #000000;
    color: #FFFFFF;
    padding: 20px;
    margin: 0;
    width: 90%;
    border-radius: 15px;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.7);
} */

.menu-section {
    background-color: #000000;
    color: #FFFFFF;
    padding: 10px;
    margin: 20px 0 20px 10%; /* Align section to the left */
    width: 60%;
    border-radius: 10px;
    box-shadow: 0px 4px 10px rgba(189, 180, 180, 0.7);
    text-align: center;
}

.menu-section h2 {
    color: #FFCC00;
    font-size: 2em;
    margin-bottom: 20px;
    cursor: pointer;
}

.menu-items {
    list-style-type: none;
    padding: 0;
    display: none; /* Hidden by default */
}

.menu-items li {
    background-color: #222222;
    color: #F7F7F7;
    padding: 10px;
    border-radius: 5px;
    margin: 10px 0;
    text-align: center;
    font-size: 1.1em;
    box-shadow: 0px 2px 5px rgba(0, 0, 0, 0.5);
}

.menu-toggle:hover {
    color: #FF9933;
    cursor: pointer;
}
</style>
</head>
<body>
<div class="content">
    <div class="header-container">
        <h1>Welcome to the<br>Roast Toast Coast Cafe!</h1>
        <p>Where flavor meets the shore!</p>
    </div>
    <section class="menu-section">
        <h2 class="menu-toggle" onclick="toggleMenuItems()">Our Menu</h2>
        <ul class="menu-items">
            <li>Breezy Brews</li>
            <li>Toasty Delights</li>
            <li>Coastal Bites</li>
            <li>Sunset Sips</li>
        </ul>
    </section>
</div>
<div class="image-container"></div>
<script>
    function toggleMenuItems() {
        const menuItems = document.querySelector('.menu-item');
        menuItems.style.display = menuItems.style.display === 'block' ? 'none' : 'block';
    }
</script>
</body>
</html>
"""

SOURCE = Path("Menu.html")
original_html = SOURCE.read_text(encoding="utf-8") if SOURCE.exists() else ORIGINAL_EMBEDDED
(OUTPUT_DIR / "Menu_original.html").write_text(original_html, encoding="utf-8")

print("Using:", "local Menu.html" if SOURCE.exists() else "embedded copy of the lab file")
print("Lines:", original_html.count("\n") + 1)

Using: embedded copy of the lab file
Lines: 141


## 2. Task 2 – Analyze the code

### IBM Bob prompt used in the lab (Code mode)
```text
Can you identify all syntax and functional errors in this HTML file:
"[your-file-path]/MenuHTML.html"?

Perform a thorough analysis.
Group the findings by severity (critical, high, medium, low).
Indicate whether each issue is related to syntax, functionality, or structure.
Identify anything that could prevent the page from rendering or behaving correctly.
```

The prompt asks for three things: a **thorough** scan, **grouping by severity**, and a **type label** (syntax / functionality / structure). The analyzer below produces exactly that output.

### Checks performed
| # | Check | Type |
|---|---|---|
| 1 | Every CSS `url(...)` is properly closed | syntax |
| 2 | Every selector used in JavaScript (`querySelector`, `getElementById` …) matches an element in the HTML | functionality |
| 3 | Every CSS selector matches at least one element in the HTML | structure |
| 4 | A `background` shorthand does not silently override an earlier `background-color` | functionality |
| 5 | Elements fetched with `querySelector` are null-checked before use | functionality |
| 6 | `<meta charset>` and `<meta viewport>` are present | structure |
| 7 | No dead (commented-out) CSS rules left in the stylesheet | structure |
| 8 | Click targets are keyboard accessible; the image has a text alternative | structure |

In [3]:
def line_of(html, snippet):
    """1-based line number of the first occurrence of `snippet` in `html`."""
    idx = html.find(snippet)
    return html.count("\n", 0, idx) + 1 if idx >= 0 else None


def strip_css_comments(css):
    return re.sub(r"/\*.*?\*/", "", css, flags=re.S)


def analyze(html):
    """Return a list of findings (dicts) for the given HTML document."""
    findings = []

    def add(severity, category, title, detail, impact, fix, snippet=None, auto=None):
        findings.append({
            "severity": severity, "category": category, "title": title,
            "detail": detail, "impact": impact, "recommended_fix": fix,
            "line": line_of(html, snippet) if snippet else None,
            "auto_fix": auto,
        })

    soup = BeautifulSoup(html, "html.parser")
    css_raw = "\n".join(s.get_text() for s in soup.find_all("style"))
    css = strip_css_comments(css_raw)
    js = "\n".join(s.get_text() for s in soup.find_all("script"))
    html_classes = {c for t in soup.find_all(True) for c in (t.get("class") or [])}

    # 1 ── unclosed url( ... ) -------------------------------------------------
    valid_url = r"url\(\s*(?:'[^']*'|\"[^\"]*\"|[^)'\"\s]*)\s*\)"
    for m in re.finditer(r"url\(", css):
        tail = css[m.start():]
        if not re.match(valid_url, tail):
            snippet = tail.split(";")[0][:70]
            add("critical", "syntax",
                "CSS url() is missing its closing parenthesis",
                "In the `.image-container` rule, `background-image: url('...'` ends with `;` "
                "but the `)` that closes url() is missing.",
                "The declaration is invalid, so the browser discards it and the image never "
                "displays. Browser error-recovery may also swallow the rules that follow it.",
                "Add the missing `)`, so the value reads `url('...');` instead of `url('...';`.",
                snippet=snippet)

    # 2 ── JavaScript selectors that match nothing -------------------------------
    for m in re.finditer(r"querySelector(?:All)?\(\s*['\"]([^'\"]+)['\"]\s*\)", js):
        sel = m.group(1)
        if soup.select(sel):
            continue
        missing = [c for c in re.findall(r"\.([\w-]+)", sel) if c not in html_classes]
        fixed_sel, hint = sel, ""
        for c in missing:
            close = difflib.get_close_matches(c, sorted(html_classes), n=1, cutoff=0.6)
            if close:
                fixed_sel = fixed_sel.replace("." + c, "." + close[0])
                hint = f" Closest class in the HTML: `.{close[0]}`."
        add("critical", "functionality",
            f"JavaScript selector '{sel}' matches no element",
            f"`document.querySelector('{sel}')` looks for a class that does not exist in the HTML.{hint}",
            "querySelector returns null, so `menuItems.style` throws a TypeError and the "
            "menu never opens when 'Our Menu' is clicked.",
            f"Use the class that exists in the markup: '{sel}' -> '{fixed_sel}'.",
            snippet=m.group(0),
            auto=(sel, fixed_sel) if fixed_sel != sel else None)

    # 3 ── CSS selectors that match nothing --------------------------------------
    for rule in re.finditer(r"([^{}]+)\{([^{}]*)\}", css):
        selector_text = rule.group(1).strip()
        if selector_text.startswith("@"):
            continue
        for sel in [s.strip() for s in selector_text.split(",")]:
            base = re.sub(r"::?[\w-]+(\([^)]*\))?", "", sel).strip()   # drop :hover etc.
            if base and not soup.select(base):
                add("high", "structure",
                    f"CSS selector '{sel}' matches no element",
                    f"The stylesheet targets `{sel}` but the HTML has no such element. "
                    "The page wraps the welcome text in `div.header-container`, not a <header> tag.",
                    "The heading and tagline styles (gold title, white tagline, text shadows) are never applied.",
                    "Re-target the rule at the real structure, e.g. `.header-container h1` / "
                    "`.header-container p` (or change the div to a semantic <header>).",
                    snippet=sel)

    # 4 ── background shorthand overriding background-color ---------------------
    for rule in re.finditer(r"([^{}]+)\{([^{}]*)\}", css):
        decls = [d.strip() for d in re.split(r";(?![^(]*\))", rule.group(2)) if d.strip()]
        names = [d.split(":")[0].strip() for d in decls]
        if "background-color" in names and "background" in names and \
                names.index("background") > names.index("background-color"):
            sel = rule.group(1).strip()
            bg = next(d for d in decls if d.split(":")[0].strip() == "background")
            add("medium", "functionality",
                f"`{bg}` overrides `background-color` in '{sel}'",
                "A later `background` shorthand resets the earlier semi-transparent "
                "`background-color`, leaving an opaque light panel.",
                "Once the heading selectors are fixed, gold and white text would sit on a "
                "light 'wheat' background and be almost unreadable.",
                "Delete the stray `background: ...;` line so the intended dark, translucent card is used.",
                snippet=bg)

    # 5 ── unguarded querySelector result ----------------------------------------
    for m in re.finditer(r"(?:const|let|var)\s+(\w+)\s*=\s*document\.querySelector\(", js):
        v = m.group(1)
        if not re.search(rf"if\s*\(\s*!\s*{v}\b|{v}\s*&&|{v}\s*\?\.", js):
            add("medium", "functionality",
                f"'{v}' is used without a null check",
                f"`{v}` comes from querySelector but is dereferenced immediately.",
                "Any future typo in a selector becomes a hard TypeError instead of a harmless no-op.",
                f"Add `if (!{v}) return;` right after the lookup.",
                snippet=m.group(0))

    # 6 ── missing meta tags -----------------------------------------------------
    missing_meta = []
    if not soup.find("meta", attrs={"charset": True}):
        missing_meta.append("charset")
    if not soup.find("meta", attrs={"name": "viewport"}):
        missing_meta.append("viewport")
    if missing_meta:
        add("medium", "structure",
            f"Missing <meta> tag(s): {', '.join(missing_meta)}",
            "The <head> contains only <title> and <style>.",
            "Without `charset` the encoding is guessed by the browser; without `viewport` mobile "
            "browsers render the two-column page zoomed-out.",
            'Add `<meta charset="UTF-8">` and `<meta name="viewport" content="width=device-width, initial-scale=1.0">`.',
            snippet="<head>")

    # 7 ── dead CSS in comments --------------------------------------------------
    for c in re.finditer(r"/\*\s*([^*]*?\{.*?\})\s*\*/", css_raw, flags=re.S):
        sel = c.group(1).split("{")[0].strip()
        add("low", "structure",
            f"Commented-out duplicate rule for '{sel}'",
            "An older version of the rule is left in a comment above the active one.",
            "No runtime effect, but it clutters the stylesheet and invites editing the wrong copy.",
            "Remove the commented block (keep it in version control instead).",
            snippet="/* " + sel)

    # 8 ── accessibility / semantics ---------------------------------------------
    for t in soup.find_all(attrs={"onclick": True}):
        if t.name not in ("button", "a") and not t.get("tabindex") and t.get("role") != "button":
            add("low", "structure",
                f"<{t.name}> used as a button is not keyboard accessible",
                f"<{t.name} class=\"{' '.join(t.get('class', []))}\"> has an inline onclick but no role, tabindex or key handler.",
                "Keyboard and screen-reader users cannot open the menu.",
                "Use a real <button> (styled like the heading) or add role=\"button\", tabindex=\"0\", aria-expanded and a keydown handler.",
                snippet="onclick=")
    for t in soup.find_all(class_="image-container"):
        if not t.get("role") and not t.get("aria-label") and not t.find("img"):
            add("low", "structure",
                "Image is a CSS background with no text alternative",
                "The cafe photo is drawn with `background-image` on an empty <div>.",
                "Screen readers ignore it; there is no alt text.",
                "Add `role=\"img\" aria-label=\"...\"` to the div or use an <img alt=\"...\"> element.",
                snippet='class="image-container"')

    # sort + number -----------------------------------------------------------
    findings.sort(key=lambda f: (SEVERITY_ORDER.index(f["severity"]), f["line"] or 0))
    for i, f in enumerate(findings, 1):
        f["id"] = f"F{i:02d}"
    return findings


findings = analyze(original_html)
print(f"{len(findings)} issues found:", dict(Counter(f["severity"] for f in findings)))

10 issues found: {'critical': 2, 'high': 2, 'medium': 3, 'low': 3}


### Task 2c – Review the analysis results (grouped by severity)

In [4]:
def show_report(findings):
    df = pd.DataFrame(findings)[["id", "severity", "category", "line", "title"]]
    df["severity"] = pd.Categorical(df["severity"], SEVERITY_ORDER, ordered=True)
    return df.sort_values(["severity", "id"]).reset_index(drop=True)


def print_details(findings):
    for sev in SEVERITY_ORDER:
        group = [f for f in findings if f["severity"] == sev]
        if not group:
            continue
        print(f"\n{'=' * 78}\n{sev.upper()}  ({len(group)})\n{'=' * 78}")
        for f in group:
            print(f"\n[{f['id']}] {f['title']}")
            print(f"    type  : {f['category']}    line: {f['line']}")
            for label in ("detail", "impact", "recommended_fix"):
                print(textwrap.fill(f[label], 74, initial_indent=f"    {label.split('_')[-1]:<7}: ",
                                    subsequent_indent=" " * 13))


show_report(findings)

,id,severity,category,line,title
0,F01,critical,syntax,32,CSS url() is missing its closing parenthesis
1,F02,critical,functionality,135,JavaScript selector '.menu-item' matches no el...
2,F03,high,structure,50,CSS selector 'header h1' matches no element
3,F04,high,structure,58,CSS selector 'header p' matches no element
4,F05,medium,structure,3,"Missing <meta> tag(s): charset, viewport"
5,F06,medium,functionality,47,`background: wheat` overrides `background-colo...
6,F07,medium,functionality,135,'menuItems' is used without a null check
7,F08,low,structure,65,Commented-out duplicate rule for '.menu-section'
8,F09,low,structure,123,<h2> used as a button is not keyboard accessible
9,F10,low,structure,132,Image is a CSS background with no text alterna...


In [5]:
print_details(findings)


CRITICAL  (2)

[F01] CSS url() is missing its closing parenthesis
    type  : syntax    line: 32
    detail : In the `.image-container` rule, `background-image: url('...'`
             ends with `;` but the `)` that closes url() is missing.
    impact : The declaration is invalid, so the browser discards it and
             the image never displays. Browser error-recovery may also
             swallow the rules that follow it.
    fix    : Add the missing `)`, so the value reads `url('...');` instead
             of `url('...';`.

[F02] JavaScript selector '.menu-item' matches no element
    type  : functionality    line: 135
    detail : `document.querySelector('.menu-item')` looks for a class that
             does not exist in the HTML. Closest class in the HTML:
             `.menu-items`.
    impact : querySelector returns null, so `menuItems.style` throws a
             TypeError and the menu never opens when 'Our Menu' is
             clicked.
    fix    : Use the class that ex

## 3. Task 3 – Apply code corrections

### IBM Bob prompt used in the lab (Code mode)
```text
Rewrite this file to fix the identified issues.
Apply the recommendations from your previous analysis.
Preserve the original structure where possible.
```

The fixer below follows the third line of that prompt: it makes **targeted, minimal edits** instead of rewriting the page, so the diff stays small and reviewable.

In [6]:
def apply_fixes(html, findings):
    """Apply targeted fixes. Returns (fixed_html, list_of_changes_applied)."""
    log = []

    def sub(pattern, repl, text, label, flags=0):
        new, n = re.subn(pattern, repl, text, flags=flags)
        if n:
            log.append(label)
        return new

    # F: close the url() parenthesis
    html = sub(r"(url\(\s*'[^']*')\s*;", r"\1);", html, "Closed the missing ')' in background-image: url(...)")
    html = sub(r'(url\(\s*"[^"]*")\s*;', r"\1);", html, "Closed the missing ')' in a url(\"...\") value")

    # F: JavaScript selector mismatch (derived from the analysis itself)
    for f in findings:
        if f.get("auto_fix"):
            old, new = f["auto_fix"]
            html = sub(rf"(querySelector(?:All)?\(\s*)(['\"]){re.escape(old)}\2",
                       rf"\1\2{new}\2", html,
                       f"JavaScript selector '{old}' -> '{new}'")

    # F: add null guard after querySelector
    if any("null check" in f["title"] for f in findings):
        html = sub(r"([ \t]*)((?:const|let|var)\s+(\w+)\s*=\s*document\.querySelector\([^)]*\);)\n",
                   lambda m: f"{m.group(1)}{m.group(2)}\n{m.group(1)}if (!{m.group(3)}) {{ return; }}\n",
                   html, "Added a null guard after querySelector")

    # F: CSS selectors that targeted a non-existent <header>
    html = sub(r"(?m)^header\s+(h1|p)\b", r".header-container \1", html,
               "Re-targeted `header h1` / `header p` to `.header-container h1` / `.header-container p`")

    # F: stray background shorthand
    html = sub(r"\n[ \t]*background:\s*wheat;[^\n]*", "", html, "Removed stray `background: wheat;` override")

    # F: meta tags
    if any("Missing <meta>" in f["title"] for f in findings):
        html = sub(r"<head>\n",
                   '<head>\n<meta charset="UTF-8">\n'
                   '<meta name="viewport" content="width=device-width, initial-scale=1.0">\n',
                   html, "Added <meta charset> and <meta viewport>")

    # F: dead commented-out rule
    html = sub(r"/\*\s*\.menu-section\s*\{.*?\}\s*\*/\s*", "", html,
               "Removed commented-out duplicate `.menu-section` rule", flags=re.S)
    return html, log


fixed_html, change_log = apply_fixes(original_html, findings)
(OUTPUT_DIR / "Menu_fixed.html").write_text(fixed_html, encoding="utf-8")

print("Changes applied:")
for i, c in enumerate(change_log, 1):
    print(f"  {i}. {c}")

Changes applied:
  1. Closed the missing ')' in background-image: url(...)
  2. JavaScript selector '.menu-item' -> '.menu-items'
  3. Added a null guard after querySelector
  4. Re-targeted `header h1` / `header p` to `.header-container h1` / `.header-container p`
  5. Removed stray `background: wheat;` override
  6. Added <meta charset> and <meta viewport>
  7. Removed commented-out duplicate `.menu-section` rule


### Review the changes – original vs. corrected (unified diff)

In [7]:
diff = difflib.unified_diff(
    original_html.splitlines(), fixed_html.splitlines(),
    fromfile="Menu.html (original)", tofile="Menu.html (corrected)", lineterm="", n=1)
print("\n".join(diff))

--- Menu.html (original)
+++ Menu.html (corrected)
@@ -3,2 +3,4 @@
 <head>
+<meta charset="UTF-8">
+<meta name="viewport" content="width=device-width, initial-scale=1.0">
 <title>Roast Toast Coast Cafe</title>
@@ -31,3 +33,3 @@
 .image-container {
-    background-image: url('https://images.unsplash.com/photo-1514066558159-fc8c737ef259?q=80&w=2574&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8MHx8fA%3D%3D';
+    background-image: url('https://images.unsplash.com/photo-1514066558159-fc8c737ef259?q=80&w=2574&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8MHx8fA%3D%3D');
     background-size: cover;
@@ -46,6 +48,5 @@
     text-align: center;
-    background: wheat;
 }
 
-header h1 {
+.header-container h1 {
     color: #FFCC00; /* Bright gold for contrast */
@@ -57,3 +58,3 @@
 
-header p {
+.header-container p {
     font-size: 1.3em;
@@ -63,12 +64,2 @@
 }
-
-/* .menu-section {
-    background-color: #000000;
-    color: #

## 4. Verification

Re-running the same analyzer on the corrected file shows whether the fixes worked. The lab says **not** to render the page, so verification is done on the code itself:

* the analyzer reports no critical, high or medium issues;
* the JavaScript selector now resolves to the real `<ul>`;
* the image `url(...)` is a valid CSS function;
* the HTML still parses and keeps the same elements as before.

In [8]:
after = analyze(fixed_html)
print("Issues before :", dict(Counter(f["severity"] for f in findings)))
print("Issues after  :", dict(Counter(f["severity"] for f in after)))

soup_after = BeautifulSoup(fixed_html, "html.parser")
js_after = "\n".join(s.get_text() for s in soup_after.find_all("script"))
selector_used = re.search(r"querySelector\('([^']+)'\)", js_after).group(1)

css_after = strip_css_comments("\n".join(x.get_text() for x in soup_after.find_all("style")))
valid_url_re = r"url\(\s*(?:'[^']*'|\"[^\"]*\"|[^)'\"\s]*)\s*\)"
url_tails = re.findall(r"url\([^\n]*", css_after)

checks = {
    "No critical/high/medium issues remain": not any(f["severity"] in ("critical", "high", "medium") for f in after),
    f"JS selector '{selector_used}' matches the menu <ul>": len(soup_after.select(selector_used)) == 1,
    f"All {len(url_tails)} CSS url(...) values are closed": all(re.match(valid_url_re, t) for t in url_tails),
    "No CSS rule still targets a missing <header>": re.search(r"(?m)^header\s", css_after) is None,
    "Same HTML elements as the original body": [t.name for t in soup_after.body.find_all(True) if t.name != "script"]
        == [t.name for t in BeautifulSoup(original_html, "html.parser").body.find_all(True) if t.name != "script"],
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "One or more verification checks failed"

print("\nRemaining (low-severity) recommendations, intentionally not auto-applied:")
for f in after:
    print(f"  [{f['id']}] {f['title']}")

Issues before : {'critical': 2, 'high': 2, 'medium': 3, 'low': 3}
Issues after  : {'low': 2}
PASS  No critical/high/medium issues remain
PASS  JS selector '.menu-items' matches the menu <ul>
PASS  All 2 CSS url(...) values are closed
PASS  No CSS rule still targets a missing <header>
PASS  Same HTML elements as the original body

Remaining (low-severity) recommendations, intentionally not auto-applied:
  [F01] <h2> used as a button is not keyboard accessible
  [F02] Image is a CSS background with no text alternative


In [9]:
# Save the machine-readable report for the project documentation
report = {"before": findings, "after": after, "changes_applied": change_log}
(OUTPUT_DIR / "findings.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved:", *[p.name for p in sorted(OUTPUT_DIR.iterdir())], sep="\n  ")

Saved:
  Menu_fixed.html
  Menu_original.html
  findings.json


## 5. Task 4 – Reflection

**Selector alignment.** The JavaScript looked for `.menu-item` while the HTML class is `.menu-items` (plural). A one-letter mismatch made `querySelector` return `null`, so the click handler threw an error and the menu never opened.

**Structure and semantics.** The CSS targeted `header h1` and `header p`, but the page has no `<header>` element; the text lives in `div.header-container`. The rules matched nothing, so the title and tagline lost their styling. Fixing the selectors (rather than rewriting the markup) preserved the original structure, as the lab prompt requires.

**Error visibility.** The hardest bugs to spot by eye were the **missing `)`** in `url(...)` (the line looks correct at a glance) and the **`background: wheat`** override, which is only harmful *after* the heading selectors are repaired. Categorising findings by severity and type surfaces such problems systematically.

**Prompt effectiveness.** The lab prompt works because it defines the scope (whole file), the output structure (severity + type) and the goal (anything that blocks rendering or behaviour). The follow-up prompt names an explicit constraint – "preserve the original structure" – which keeps the fix small and reviewable.

### Key takeaway
Tiny syntax or naming inconsistencies can block an entire feature. A structured prompt, a categorised report and a before/after diff turn troubleshooting from guesswork into a repeatable process.

### How to run
```bash
pip install -r requirements.txt
jupyter notebook YourName_TroubleshootYourCode.ipynb   # Run > Run All Cells
```
Outputs are written to the `output/` folder: `Menu_original.html`, `Menu_fixed.html`, `findings.json`.